# ExamGuard AI — exploration des données et des résultats

Ce notebook n'entraîne rien : il relit les données et les résultats produits par
`python -m examguard_ml.pipeline` (à lancer d'abord depuis le dossier `ml/`).

In [ ]:
import sys, json
from pathlib import Path
ML = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path[:0] = [str(ML / 'src'), str(ML)]
import pandas as pd
from examguard_ml import data, config
from examguard_ml.features import browser_features, webcam_features

## 1. Données (contrat d'entrée brut puis variables construites)

In [ ]:
b = data.prepare_browser()
w = data.prepare_webcam()
print('B :', b.X.shape, 'positifs', b.y.sum(), '| A :', w.X.shape, 'positifs', w.y.sum())
display(browser_features(b.X).describe().T)
display(webcam_features(w.X).describe().T)

In [ ]:
# Les 77 tricheurs ont tous réussi l'examen
pd.crosstab(b.full['result'], b.full['prediction'])

## 2. Résultats des modèles

In [ ]:
from examguard_ml.train import comparison_table
for br in ('browser', 'webcam'):
    res = json.loads((config.REPORTS / f'results_{br}.json').read_text(encoding='utf-8'))
    print(res['title'])
    display(comparison_table(res))

## 3. Contrat d'interface

In [ ]:
import inference
session = {'exam_mode': 'remote', 'tot_question': 20, 'tot_que_attempted': 19,
           'que_att_within_idl_time': 18, 'idl_time_correct_que': 17, 'tot_correct_que': 18,
           'exam_finish_time_in_sec': 900}
pb = inference.predict_browser(session)
pa = inference.predict_webcam_session([{k: float(v) if k not in ('face_present','no_of_face','hand_count','phone_present') else int(v)
                                        for k, v in w.X.iloc[i].items()} for i in range(10)])
print(json.dumps(pb, indent=2, ensure_ascii=False))
print(json.dumps(inference.fuse(pa['probabilite_deploiement'], pb['probabilite_deploiement']), indent=2, ensure_ascii=False))
print(json.dumps(inference.explain('browser', session, k=3), indent=2, ensure_ascii=False))